In [1]:
# https://huggingface.co/datasets/gbharti/finance-alpaca

In [2]:
! pip install -q torch torchtext transformers pandas tqdm datasets

In [3]:
from datasets import load_dataset
import pandas as pd
import time
from tqdm import tqdm

/Users/psykick/Documents/GitHub/fine-tuning/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [21]:
# load the dataset from huggingface
data_sample = load_dataset("gbharti/finance-alpaca")

# print the dataset
# the data says "test" but it's fine. We'll manage it later
data_sample

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output', 'text'],
        num_rows: 68912
    })
})

In [22]:
# converting the dataset to a pandas dataframe, keeping only the relevant columns
n = 40    # number of datapoints to load
data = [{'instruction': item['instruction'], 'output': item['output']} for item in data_sample['train']][:n]
df = pd.DataFrame(data)

del(data_sample)

df.head()   

,instruction,output
0,"For a car, what scams can be plotted with 0% f...",The car deal makes money 3 ways. If you pay in...
1,Why does it matter if a Central Bank has a neg...,"That is kind of the point, one of the hopes is..."
2,Where should I be investing my money?,"Pay off your debt. As you witnessed, no ""inve..."
3,Specifically when do options expire?,"Equity options, at least those traded in the A..."
4,Negative Balance from Automatic Options Exerci...,"Automatic exercisions can be extremely risky, ..."


In [6]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer
import torch
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split

In [7]:
# set the device (CPU is not preferred since it will be extremely slow)
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    try:
        device = torch.device("mps")
    except:
        device = torch.device("cpu")

print(device)

mps


In [8]:
# define the tokenizer and model
tokenizer = GPT2Tokenizer.from_pretrained('distilgpt2')
model = GPT2LMHeadModel.from_pretrained('distilgpt2').to(device)

# since gpt2 doesn't have a pad token, we'll use the eos token as the pad token
tokenizer.pad_token = tokenizer.eos_token


In [9]:
# visualize the model architecture
print(model)


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-5): 6 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)


In [10]:
# defining model parameters
BATCH_SIZE = 8
LEARNING_RATE = 1e-5
NUM_EPOCHS = 1

In [11]:
# dataset preparation

class AlpacaDataset(Dataset):
    def __init__(self, df, tokenizer):
        self.df = df
        self.columns = self.df.columns
        self.data = df.to_dict(orient='records')
        self.tokenizer = tokenizer
        # self.max_length = self.max_sequence_length()
        self.max_length = 256
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        x = self.data[idx][self.columns[0]]
        y = self.data[idx][self.columns[1]]
        
        text = f"{x} | {y}"
        tokens = self.tokenizer.encode_plus(
            text, return_tensors='pt', max_length=self.max_length, padding='max_length', truncation=True
        )
        return tokens

    def max_sequence_length(self):
        max_length = max(len(max(self.df[self.columns[0]], key=len)), len(max(self.df[self.columns[1]], key=len)))
        x = 2
        while x < max_length:
            x *= 2
        return min(x, 256)
    
data_sample = AlpacaDataset(df, tokenizer)

In [12]:
# check the type of the dataset
print(type(data_sample))

<class '__main__.AlpacaDataset'>


In [13]:
# creating train and test splits
train_size = int(0.8 * len(data_sample))
valid_size = len(data_sample) - train_size
train_data, valid_data = random_split(data_sample, [train_size, valid_size])

In [14]:
# create dataloaders
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_data, batch_size=BATCH_SIZE)

In [15]:
# setting the optimizer (we will use the default loss function provided by the model)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

In [16]:
# initiating a results dataframe
results = pd.DataFrame(columns=[
    'epoch', 'train_loss', 'val_loss', 'time_taken'
])

In [20]:
# training and validation loop

for epoch in range(NUM_EPOCHS):

    start_time = time.time()

    # training
    model.train()   # set the model to training mode
    epoch_training_loss = 0
    train_iterator = tqdm(train_loader, desc=f"Training Epoch {epoch+1}/{NUM_EPOCHS}")

    for batch in train_iterator:
        optimizer.zero_grad()
        inputs = batch['input_ids'].squeeze(1).to(device)
        targets = inputs.clone()
        outputs = model(input_ids=inputs, labels=targets)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        train_iterator.set_postfix({'Training loss': loss.item()})
        epoch_training_loss += loss.item()
    avg_epoch_training_loss = epoch_training_loss / len(train_iterator)

    # validation
    model.eval()    # set the model to evaluation mode
    epoch_validation_loss = 0
    val_iterator = tqdm(valid_loader, desc=f"Validation Epoch {epoch+1}/{NUM_EPOCHS}")

    with torch.no_grad():
        for batch in val_iterator:
            inputs = batch['input_ids'].squeeze(1).to(device)
            targets = inputs.clone()
            outputs = model(input_ids=inputs, labels=targets)
            loss = outputs.loss
            epoch_validation_loss += loss.item()
            val_iterator.set_postfix({'Validation loss': loss.item()})
    avg_epoch_validation_loss = epoch_validation_loss / len(val_iterator)

    end_time = time.time()
    time_taken = end_time - start_time

    new_row = {
        'epoch': epoch+1,
        'train_loss': avg_epoch_training_loss,
        'val_loss': avg_epoch_validation_loss,
        'time_taken': time_taken
    }
    results.loc[len(results)] = new_row
    
    print(f"Epoch {epoch+1} completed in {time_taken:.2f}s; validation loss: {avg_epoch_validation_loss:.4f}")

Validation Epoch 1/1: 100%|██████████| 1/1 [00:03<00:00,  3.82s/it, Validation loss=4.94]


Epoch 1 completed in 35.08s; validation loss: 4.9437


In [23]:
df.loc[0, 'instruction']

'For a car, what scams can be plotted with 0% financing vs rebate?'

In [26]:
input_str = "For a car, what scams can be plotted with 0% financing vs rebate?"
input_ids = tokenizer.encode(input_str, return_tensors='pt').to(device)

output = model.generate(
    input_ids,
    max_length=256,
    num_return_sequences=1,
    do_sample=True,
    top_k=8,
    top_p=0.95,
    temperature=0.5,
    repetition_penalty=1.2
)

decoded_output = tokenizer.decode(output[0], skip_special_tokens=True)
print(decoded_output)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


For a car, what scams can be plotted with 0% financing vs rebate?
I think it's possible to calculate the amount of money that you are paying for your vehicle. It is not easy as there may be some fees or credits on each purchase but if they come from an affiliate who makes them in exchange for their products and services we will do our best to make sure all these items go through when buying!
